In [ ]:

# Importamos librerías

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Importamos dataset
# https://www.kaggle.com/code/syeddanish5/genetic-disease-prediction-model

data = pd.read_csv(
    r"C:/Users/karen/Downloads/genetic_disease_dataset.csv"
)

target = "Disease"

seed = 42

k_folds = 5
# Al dividir los datos en grupos, quedamos con pocas observaciones
# por grupo, así que conviene usar validación cruzada.


In [ ]:
# Tenemos 5 categorías de enfermedad.
# Asignamos a cada código el nombre de la patología que le corresponde.

MAPEO_ENFERMEDADES = {
    0: "Talasemia",
    1: "Hemofilia",
    2: "Cancer_de_mama",
    3: "Anemia_falciforme",
    4: "Fibrosis_quistica",
}

data["Disease"] = data["Disease"].map(MAPEO_ENFERMEDADES)

In [ ]:

# Revisar observaciones por enfermedad

print(data["Disease"].value_counts())

# Las clases están balanceadas

# Revisamos valores nulos

print(data.isnull().sum())

# No hay valores nulos


In [ ]:
# Definimos los modelos a comparar

modelos = {

    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        random_state=seed,
        n_jobs=-1
    ),

    "RegresionLogistica": Pipeline([
        ("scaler", StandardScaler()),
        ("Clf", LogisticRegression(max_iter=2000))
    ]),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=seed
    )

}


In [ ]:

# Preparación de datos

def preparar_datos(data, target):

    X = data.drop(columns=[target])

    y = data[target]

    return X, y


X, y = preparar_datos(data, target)



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=seed,
    stratify=y
)

print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

In [ ]:
def comparar_modelos(modelos, X_train, y_train):

    cv = StratifiedKFold(
        n_splits=k_folds,
        shuffle=True,
        random_state=seed
    )

    resultados = []

    for nombre, modelo in modelos.items():

        scores = cross_validate(
            modelo,
            X_train,
            y_train,
            cv=cv,
            scoring="accuracy",
            n_jobs=-1
        )

        resultados.append({

            "Modelo": nombre,

            "Accuracy_media":
                scores["test_score"].mean(),

            "Accuracy_std":
                scores["test_score"].std()
        })

    df_resultados = (
        pd.DataFrame(resultados)
        .sort_values(
            "Accuracy_media",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return df_resultados


In [ ]:
df_resultados = comparar_modelos(
    modelos,
    X_train,
    y_train
)

print(df_resultados)


In [ ]:
df_sorted = df_resultados.sort_values(
    "Accuracy_media"
)

plt.figure(figsize=(8, 5))

plt.barh(
    df_sorted["Modelo"],
    df_sorted["Accuracy_media"]
)

plt.xlabel("Accuracy")
plt.title(
    "Comparación de modelos - Validación cruzada"
)

plt.xlim(0, 1)

plt.tight_layout()

plt.show()


In [ ]:

# Seleccionamos automáticamente el modelo con mayor Accuracy media

mejor_nombre = df_resultados.iloc[0]["Modelo"]

print("Modelo seleccionado:", mejor_nombre)

mejor_modelo = modelos[mejor_nombre]


In [ ]:
# Entrenamos el modelo seleccionado utilizando TRAIN completo

mejor_modelo.fit(
    X_train,
    y_train
)

# Realizamos las predicciones sobre TEST

y_pred = mejor_modelo.predict(
    X_test
)

# Calculamos Accuracy final sobre TEST

accuracy_test = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Accuracy final en TEST: {accuracy_test:.4f}"
)

# Reporte de clasificación

print("Reporte de clasificación:")

print(
    classification_report(
        y_test,
        y_pred
    )
)


In [ ]:

# Matriz de confusión

matriz = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(7, 5))

sns.heatmap(
    matriz,
    annot=True,
    fmt="d",
    xticklabels=sorted(y.unique()),
    yticklabels=sorted(y.unique())
)

plt.xlabel("Predicción")

plt.ylabel("Valor real")

plt.title(
    f"Matriz de confusión - {mejor_nombre}"
)

plt.tight_layout()

plt.show()


# Guardamos el modelo final

joblib.dump(
    mejor_modelo,
    "mejor_modelo_genetic_disease.pkl"
)

print("Modelo guardado correctamente.")

In [ ]:
#import os

#print(os.getcwd())

In [ ]:

#ruta = os.path.abspath("mejor_modelo_genetic_disease.pkl")

#rint(ruta)